# QUAL-005 - Notebook final de entrenamiento

Entrena ambos modelos finales con split por paciente y checkpoints compatibles con el runtime del repo.

- Sagital SPIDER: `SagittalUNet2D`, 4 clases, `label_group_mapping` crudo -> agrupado.
- Axial ALKAFRI/Sudirman: `AxialUNet2D`, 6 clases `[background_250, raw_0, raw_50, raw_100, raw_150, raw_200]`.
- Objetivo: Dice macro foreground >= 0.70 en test held-out por paciente.

Este notebook se escribe para Colab/Drive. No se ejecuta en local y no guarda `.pt` ni datasets en Git.

## 0. Bootstrap Colab / repo

Esta celda debe correr antes de importar `pfi_ai_service`. Si el repo no existe en `PFI_REPO_ROOT`, lo clona desde `PFI_REPO_URL`. Si ya existe, solo verifica que `model_architectures.py` este disponible y agrega `ai_service/` al `sys.path`.

In [3]:
import os
import subprocess
import sys
from pathlib import Path

PFI_REPO_URL = os.getenv("PFI_REPO_URL", "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git")
PFI_REPO_ROOT = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule"))

if not PFI_REPO_ROOT.exists():
    print(f"Clonando AI Module en {PFI_REPO_ROOT} desde {PFI_REPO_URL}")
    subprocess.run(["git", "clone", PFI_REPO_URL, str(PFI_REPO_ROOT)], check=True)
else:
    print(f"AI Module encontrado en {PFI_REPO_ROOT}")

arch_path = PFI_REPO_ROOT / "ai_service" / "pfi_ai_service" / "model_architectures.py"
if not arch_path.exists():
    raise FileNotFoundError(f"No existe model_architectures.py en {arch_path}")

ai_service_path = str(PFI_REPO_ROOT / "ai_service")
if ai_service_path not in sys.path:
    sys.path.insert(0, ai_service_path)
print("PYTHONPATH ai_service OK:", ai_service_path)

Clonando AI Module en /content/PFI_MVPTest_Enzo_AImodule desde https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git
PYTHONPATH ai_service OK: /content/PFI_MVPTest_Enzo_AImodule/ai_service


In [4]:
from __future__ import annotations
import json, os, random, sys, time
from dataclasses import asdict, dataclass
from hashlib import sha256
from pathlib import Path
from typing import Any, Iterable, Literal

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageEnhance
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
try:
    import SimpleITK as sitk
except Exception:
    sitk = None
try:
    import pydicom
except Exception:
    pydicom = None

# En Colab, montar Drive si hace falta:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:
@dataclass
class TrainConfig:
    repo_root: Path = PFI_REPO_ROOT
    output_dir: Path = Path(os.getenv("PFI_TRAIN_OUTPUT_DIR", "/content/drive/MyDrive/PFI_MVP/models/final_training"))
    target_size: tuple[int, int] = (256, 256)
    base_channels: int = int(os.getenv("PFI_BASE_CHANNELS", "16"))  # usar 32 si la GPU lo permite
    batch_size: int = int(os.getenv("PFI_BATCH_SIZE", "8"))
    max_epochs: int = int(os.getenv("PFI_MAX_EPOCHS", "80"))
    patience: int = int(os.getenv("PFI_EARLY_STOP_PATIENCE", "12"))
    lr: float = float(os.getenv("PFI_LR", "0.0008"))
    seed: int = int(os.getenv("PFI_SEED", "20260716"))
    num_workers: int = int(os.getenv("PFI_NUM_WORKERS", "2"))

    spider_images_dir: Path = Path(os.getenv("SPIDER_IMAGES_DIR", "/content/drive/MyDrive/PFI_MVP/data/SPIDER/images"))
    spider_masks_dir: Path = Path(os.getenv("SPIDER_MASKS_DIR", "/content/drive/MyDrive/PFI_MVP/data/SPIDER/masks"))
    spider_label_map_json: Path = Path(os.getenv("SPIDER_LABEL_GROUP_MAPPING_JSON", "/content/drive/MyDrive/PFI_MVP/results/E5_multiclase_agrupado/E5_multiclass_label_mapping.json"))
    spider_checkpoint_for_label_map: Path = Path(os.getenv("SPIDER_CHECKPOINT_FOR_LABEL_MAP", "/content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best.pt"))

    axial_split_csv: Path = Path(os.getenv("AXIAL_E9_CURATED_SPLIT_CSV", "/content/drive/MyDrive/PFI_MVP/results/E9_alkafri_axial_t2_final_labels_baseline/E9_t2_final_labels_curated_split.csv"))
    axial_images_dir: Path = Path(os.getenv("AXIAL_IMAGES_DIR", "/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI"))
    axial_masks_dir: Path = Path(os.getenv("AXIAL_MASKS_DIR", "/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI"))
    axial_image_col: str = os.getenv("AXIAL_IMAGE_COL", "image_file_path")
    axial_mask_col: str = os.getenv("AXIAL_MASK_COL", "final_label_file_path")
    axial_patient_col: str = os.getenv("AXIAL_PATIENT_COL", "case_id_norm")
    axial_split_col: str = os.getenv("AXIAL_SPLIT_COL", "split")
    require_gpu: bool = os.getenv("REQUIRE_GPU", "1") == "1"
    axial_raw_pairing_mode: bool = os.getenv("AXIAL_RAW_PAIRING_MODE", "0") == "1"

CFG = TrainConfig(); CFG.output_dir.mkdir(parents=True, exist_ok=True)
print(CFG)

def set_seed(seed:int)->None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
set_seed(CFG.seed)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
if CFG.require_gpu and DEVICE.type != "cuda":
    raise RuntimeError("REQUIRE_GPU=1 pero CUDA no esta disponible. Frenando entrenamiento final para evitar corrida CPU accidental.")
print("device", DEVICE)

sys.path.insert(0, str(CFG.repo_root / "ai_service"))
from pfi_ai_service.model_architectures import AxialUNet2D, SagittalUNet2D, build_checkpoint_model

## Labels, metricas y helpers

El sagital exige un `label_group_mapping` real desde Drive. Si falta, el notebook falla: no se inventan mappings. El axial E9 curado debe venir en labels `0..5`; si se usa pairing crudo, activar `AXIAL_RAW_PAIRING_MODE=1` para mapear `{250:0,0:1,50:2,100:3,150:4,200:5}`.

In [ ]:
SAGITTAL_CLASSES=["background","vertebra_group","canal","disc_group"]
AXIAL_CLASSES=["background_250","raw_0","raw_50","raw_100","raw_150","raw_200"]
AXIAL_RAW_LABEL_MAP={250:0,0:1,50:2,100:3,150:4,200:5}

def load_json_mapping(path:Path, checkpoint_fallback:Path|None=None)->dict[int,int]:
    if path.exists():
        data=json.loads(path.read_text())
        out={int(k):int(v) for k,v in data.items()}; out.setdefault(0,0); return out
    if checkpoint_fallback is not None and checkpoint_fallback.exists():
        ckpt=torch.load(checkpoint_fallback,map_location="cpu",weights_only=False)
        mapping=ckpt.get("label_group_mapping") if isinstance(ckpt,dict) else None
        if mapping:
            out={int(k):int(v) for k,v in mapping.items()}; out.setdefault(0,0); return out
    raise FileNotFoundError(f"Falta label_group_mapping real: {path}; fallback={checkpoint_fallback}")

def apply_label_map(mask:np.ndarray, label_map:dict[int,int]|None, num_classes:int, case_id:str)->np.ndarray:
    mask=np.asarray(mask)
    if label_map is None:
        labels=sorted(int(v) for v in np.unique(mask)); bad=[v for v in labels if v<0 or v>=num_classes]
        if bad: raise ValueError(f"{case_id}: labels fuera de rango {bad}; proveer mapping")
        return mask.astype(np.int64)
    labels=sorted(int(v) for v in np.unique(mask)); missing=[v for v in labels if v not in label_map]
    if missing: raise ValueError(f"{case_id}: labels sin mapping {missing[:20]}")
    out=np.zeros(mask.shape,dtype=np.int64)
    for raw,cls in label_map.items():
        if cls<0 or cls>=num_classes: raise ValueError(f"class_id fuera de rango: {cls}")
        out[mask==raw]=cls
    return out

def read_array(path:Path)->np.ndarray:
    s=path.suffix.lower()
    if s==".npy": return np.load(path)
    if s in {".png",".jpg",".jpeg",".bmp",".tif",".tiff"}: return np.asarray(Image.open(path).convert("L"))
    if s in {".mha",".mhd"}:
        if sitk is None: raise ImportError("SimpleITK requerido")
        return sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
    if s in {".dcm",".ima"}:
        if pydicom is None: raise ImportError("pydicom requerido para DICOM/IMA")
        ds=pydicom.dcmread(str(path),force=True)
        return np.asarray(ds.pixel_array)
    raise ValueError(f"Formato no soportado: {path}")

def robust_normalize(x:np.ndarray)->np.ndarray:
    x=np.asarray(x,dtype=np.float32); finite=np.isfinite(x)
    if not finite.any(): return np.zeros_like(x,dtype=np.float32)
    lo,hi=np.percentile(x[finite],[1,99])
    if hi<=lo: return np.zeros_like(x,dtype=np.float32)
    return ((np.clip(x,lo,hi)-lo)/(hi-lo+1e-8)).astype(np.float32)

def resize_pair(image:np.ndarray, mask:np.ndarray, size:tuple[int,int])->tuple[np.ndarray,np.ndarray]:
    img=(robust_normalize(image)*255).clip(0,255).astype(np.uint8)
    img=Image.fromarray(img).resize((size[1],size[0]),Image.Resampling.BILINEAR)
    msk=Image.fromarray(mask.astype(np.int32),mode="I").resize((size[1],size[0]),Image.Resampling.NEAREST)
    return np.asarray(img,dtype=np.float32)/255.0, np.asarray(msk,dtype=np.int64)

def augment(image:np.ndarray, mask:np.ndarray)->tuple[np.ndarray,np.ndarray]:
    if random.random()<0.5: image,mask=np.fliplr(image).copy(),np.fliplr(mask).copy()
    if random.random()<0.2: image,mask=np.flipud(image).copy(),np.flipud(mask).copy()
    if random.random()<0.4:
        angle=random.uniform(-8,8)
        image=np.asarray(Image.fromarray((image*255).astype(np.uint8)).rotate(angle,Image.Resampling.BILINEAR),dtype=np.float32)/255.0
        mask=np.asarray(Image.fromarray(mask.astype(np.int32),mode="I").rotate(angle,Image.Resampling.NEAREST),dtype=np.int64)
    if random.random()<0.5:
        pil=Image.fromarray((image*255).clip(0,255).astype(np.uint8))
        image=np.asarray(ImageEnhance.Contrast(pil).enhance(random.uniform(0.85,1.15)),dtype=np.float32)/255.0
    return image,mask

def patient_id_from_stem(stem:str)->str:
    return stem.replace("-","_").split("_")[0] or stem

def split_patients(ids:Iterable[str], seed:int)->dict[str,str]:
    ids=sorted(set(ids)); train,tmp=train_test_split(ids,test_size=0.30,random_state=seed)
    val,test=train_test_split(tmp,test_size=0.50,random_state=seed)
    m={x:"train" for x in train}; m.update({x:"val" for x in val}); m.update({x:"test" for x in test}); return m

## Datasets 2D y split por paciente

El split se hace por paciente/caso antes de expandir a slices. El test queda congelado en CSV dentro de `CFG.output_dir`.

In [ ]:
@dataclass(frozen=True)
class SliceRecord:
    image_path:str; mask_path:str; patient_id:str; split:Literal["train","val","test"]; plane:Literal["sagittal","axial"]; slice_index:int|None=None; slice_axis:int|None=None

class Segmentation2DDataset(Dataset):
    def __init__(self, records:list[SliceRecord], num_classes:int, label_map:dict[int,int]|None, augment_on:bool):
        self.records=records; self.num_classes=num_classes; self.label_map=label_map; self.augment_on=augment_on
    def __len__(self): return len(self.records)
    def __getitem__(self, idx:int):
        r=self.records[idx]; image=read_array(Path(r.image_path)); mask=read_array(Path(r.mask_path))
        if image.ndim==3:
            axis=r.slice_axis if r.slice_axis is not None else int(np.argmin(image.shape))
            si=r.slice_index if r.slice_index is not None else image.shape[axis]//2
            image=np.take(image,si,axis=axis); mask=np.take(mask,si,axis=axis)
        mask=apply_label_map(mask,self.label_map,self.num_classes,r.patient_id)
        image,mask=resize_pair(image,mask,CFG.target_size)
        if self.augment_on: image,mask=augment(image,mask)
        return torch.from_numpy(image[None]).float(), torch.from_numpy(mask).long()

def pair_by_stem(images_dir:Path,masks_dir:Path)->list[tuple[Path,Path,str]]:
    allowed={".mha",".mhd",".npy",".png",".tif",".tiff",".dcm",".ima"}
    images=[p for p in images_dir.rglob("*") if p.is_file() and p.suffix.lower() in allowed]
    masks=[p for p in masks_dir.rglob("*") if p.is_file() and p.suffix.lower() in allowed]
    mask_by={p.stem.replace("_mask","").replace("_label",""):p for p in masks}
    pairs=[]
    for img in sorted(images):
        key=img.stem.replace("_image","")
        if key in mask_by: pairs.append((img,mask_by[key],patient_id_from_stem(key)))
    if not pairs: raise FileNotFoundError(f"Sin pares en {images_dir} / {masks_dir}")
    return pairs

def foreground_indices(mask:np.ndarray, axis:int, max_slices:int=16, bg_ratio:float=0.20)->list[int]:
    if mask.ndim==2: return [0]
    count=mask.shape[axis]; fg=[i for i in range(count) if np.take(mask,i,axis=axis).max()>0]
    if len(fg)>max_slices: fg=[fg[i] for i in np.linspace(0,len(fg)-1,max_slices,dtype=int)]
    bg_pool=[i for i in range(count) if i not in set(fg)]
    n_bg=min(len(bg_pool),max(1,int(len(fg)*bg_ratio))) if fg else min(len(bg_pool),4)
    bg=random.sample(bg_pool,n_bg) if n_bg else []
    return sorted(set(fg+bg))

def build_spider_records()->tuple[list[SliceRecord],dict[int,int]]:
    label_map=load_json_mapping(CFG.spider_label_map_json, CFG.spider_checkpoint_for_label_map)
    pairs=pair_by_stem(CFG.spider_images_dir,CFG.spider_masks_dir)
    p_split=split_patients([pid for _,_,pid in pairs],CFG.seed)
    records=[]
    for img,msk,pid in pairs:
        raw=read_array(msk); grouped=apply_label_map(raw,label_map,4,pid)
        axis=0 if grouped.ndim==2 else int(np.argmin(grouped.shape))
        for si in foreground_indices(grouped,axis): records.append(SliceRecord(str(img),str(msk),pid,p_split[pid],"sagittal",si,axis))
    return records,label_map

def resolve_path(v:Any,bases:list[Path])->Path:
    p=Path(str(v))
    if p.is_absolute() and p.exists(): return p
    for b in bases:
        c=b/p
        if c.exists(): return c
    return p

def build_axial_records()->tuple[list[SliceRecord],dict[int,int]|None]:
    if not CFG.axial_split_csv.exists(): raise FileNotFoundError(f"Falta split axial E9: {CFG.axial_split_csv}")
    df=pd.read_csv(CFG.axial_split_csv)
    for col in [CFG.axial_image_col,CFG.axial_mask_col]:
        if col not in df.columns: raise KeyError(f"Falta columna {col}; disponibles {list(df.columns)}")
    if CFG.axial_patient_col not in df.columns: df[CFG.axial_patient_col]=df[CFG.axial_image_col].map(lambda x: patient_id_from_stem(Path(str(x)).stem))
    if CFG.axial_split_col not in df.columns:
        p_split=split_patients(df[CFG.axial_patient_col].astype(str),CFG.seed); df[CFG.axial_split_col]=df[CFG.axial_patient_col].astype(str).map(p_split)
    df[CFG.axial_split_col]=df[CFG.axial_split_col].astype(str).str.lower()
    split_counts=df[CFG.axial_split_col].value_counts().to_dict()
    for required_split in ["train","val","test"]:
        if split_counts.get(required_split,0)==0:
            raise ValueError(f"Split axial vacio: {required_split}; counts={split_counts}")
    bases=[CFG.axial_images_dir,CFG.axial_masks_dir,CFG.axial_split_csv.parent]
    records=[]
    for row in df.itertuples(index=False):
        img=resolve_path(getattr(row,CFG.axial_image_col),bases); msk=resolve_path(getattr(row,CFG.axial_mask_col),bases)
        if not img.exists() or not msk.exists(): raise FileNotFoundError(f"Path axial inexistente: {img} / {msk}")
        records.append(SliceRecord(str(img),str(msk),str(getattr(row,CFG.axial_patient_col)),str(getattr(row,CFG.axial_split_col)),"axial"))
    return records, (AXIAL_RAW_LABEL_MAP if CFG.axial_raw_pairing_mode else None)

def freeze_records(records:list[SliceRecord], path:Path)->None:
    path.parent.mkdir(parents=True,exist_ok=True); pd.DataFrame([asdict(r) for r in records]).to_csv(path,index=False); print(path)

def summarize_records(records:list[SliceRecord], title:str):
    df=pd.DataFrame([asdict(r) for r in records]); print(title); print(df.groupby("split").agg(n_slices=("image_path","count"),n_patients=("patient_id","nunique"))); return df

## Preflight obligatorio

Valida repo, arquitectura, rutas SPIDER, mapping sagital, CSV axial E9, columnas reales, primeros paths image/mask y splits train/val/test no vacios antes de entrenar.

In [ ]:
def preflight() -> None:
    print("== Repo ==")
    if not CFG.repo_root.exists():
        raise FileNotFoundError(f"repo_root no existe: {CFG.repo_root}")
    arch = CFG.repo_root / "ai_service" / "pfi_ai_service" / "model_architectures.py"
    if not arch.exists():
        raise FileNotFoundError(f"model_architectures.py no existe: {arch}")
    print("OK", arch)

    print("== GPU ==")
    if CFG.require_gpu and not torch.cuda.is_available():
        raise RuntimeError("REQUIRE_GPU=1 pero CUDA no esta disponible")
    print("CUDA disponible:", torch.cuda.is_available())

    print("== SPIDER ==")
    if not CFG.spider_images_dir.exists():
        raise FileNotFoundError(f"SPIDER_IMAGES_DIR no existe: {CFG.spider_images_dir}")
    if not CFG.spider_masks_dir.exists():
        raise FileNotFoundError(f"SPIDER_MASKS_DIR no existe: {CFG.spider_masks_dir}")
    mapping = load_json_mapping(CFG.spider_label_map_json, CFG.spider_checkpoint_for_label_map)
    print("SPIDER mapping labels:", sorted(mapping.items())[:10], "... total", len(mapping))

    print("== Axial E9 ==")
    if not CFG.axial_split_csv.exists():
        raise FileNotFoundError(f"CSV axial no existe: {CFG.axial_split_csv}")
    df = pd.read_csv(CFG.axial_split_csv)
    required_cols = {CFG.axial_image_col, CFG.axial_mask_col, CFG.axial_patient_col, CFG.axial_split_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise KeyError(f"Columnas axiales faltantes: {missing}; columnas reales: {list(df.columns)}")
    counts = df[CFG.axial_split_col].astype(str).str.lower().value_counts().to_dict()
    for split_name in ["train", "val", "test"]:
        if counts.get(split_name, 0) == 0:
            raise ValueError(f"Split axial vacio: {split_name}; counts={counts}")
    bases = [CFG.axial_images_dir, CFG.axial_masks_dir, CFG.axial_split_csv.parent]
    sample = df.iloc[0]
    image_path = resolve_path(sample[CFG.axial_image_col], bases)
    mask_path = resolve_path(sample[CFG.axial_mask_col], bases)
    if not image_path.exists():
        raise FileNotFoundError(f"Primer image_file_path no existe: {image_path}")
    if not mask_path.exists():
        raise FileNotFoundError(f"Primer final_label_file_path no existe: {mask_path}")
    print("Axial splits:", counts)
    print("Primer image/mask OK:", image_path, mask_path)

preflight()

## Loss, pesos por clase, entrenamiento y guardado compatible runtime

El checkpoint se guarda como dict con `model_state_dict`, `num_classes`, `base_channels`, `target_size`, `class_weights`, `metrics`, y metadata de labels. El axial se guarda con metadata, no como state_dict crudo.

In [ ]:
def dice_loss(logits:torch.Tensor,target:torch.Tensor,num_classes:int,eps:float=1e-6)->torch.Tensor:
    probs=torch.softmax(logits,dim=1); oh=F.one_hot(target,num_classes).permute(0,3,1,2).float()
    inter=(probs*oh).sum((0,2,3)); den=(probs+oh).sum((0,2,3)); dice=(2*inter+eps)/(den+eps)
    return 1-dice[1:].mean()

def estimate_weights(records:list[SliceRecord],num_classes:int,label_map:dict[int,int]|None,boost:dict[int,float]|None=None,max_records:int=512)->torch.Tensor:
    counts=np.ones(num_classes,dtype=np.float64); sample=records if len(records)<=max_records else random.sample(records,max_records)
    ds=Segmentation2DDataset(sample,num_classes,label_map,False)
    for _,mask in ds:
        vals,freq=torch.unique(mask,return_counts=True)
        for v,f in zip(vals.tolist(),freq.tolist()): counts[int(v)]+=int(f)
    w=(counts.sum()/counts); w=w/w.mean()
    if boost:
        for cid,factor in boost.items(): w[cid]*=factor
    return torch.tensor(np.clip(w,0.25,8.0),dtype=torch.float32)

def build_model(plane:str,num_classes:int,base_channels:int)->nn.Module:
    return SagittalUNet2D(num_classes=num_classes,base_channels=base_channels) if plane=="sagittal" else AxialUNet2D(num_classes=num_classes,base_channels=base_channels)

def evaluate(model:nn.Module,loader:DataLoader,num_classes:int)->dict[str,Any]:
    model.eval(); acc={c:{"intersection":0,"pred":0,"gt":0,"union":0,"gt_present_cases":0,"pred_present_cases":0} for c in range(num_classes)}
    with torch.inference_mode():
        for x,y in loader:
            pred=torch.argmax(torch.softmax(model(x.to(DEVICE)),dim=1),dim=1).cpu().numpy(); gt=y.numpy()
            for b in range(pred.shape[0]):
                for c in range(num_classes):
                    p=pred[b]==c; g=gt[b]==c
                    acc[c]["intersection"]+=int(np.logical_and(p,g).sum()); acc[c]["pred"]+=int(p.sum()); acc[c]["gt"]+=int(g.sum()); acc[c]["union"]+=int(np.logical_or(p,g).sum())
                    acc[c]["gt_present_cases"]+=int(g.any()); acc[c]["pred_present_cases"]+=int(p.any())
    per={}; dice_fg=[]; iou_fg=[]; dice_ex_raw0=[]
    for c,s in acc.items():
        dice=None if s["pred"]+s["gt"]==0 else 2*s["intersection"]/(s["pred"]+s["gt"])
        iou=None if s["union"]==0 else s["intersection"]/s["union"]
        per[str(c)]={**s,"dice":dice,"iou":iou}
        if c!=0 and dice is not None: dice_fg.append(dice)
        if c!=0 and iou is not None: iou_fg.append(iou)
        if c not in {0,1} and dice is not None: dice_ex_raw0.append(dice)
    return {"per_class":per,"dice_macro_no_bg":float(np.mean(dice_fg)) if dice_fg else None,"iou_macro_no_bg":float(np.mean(iou_fg)) if iou_fg else None,"dice_macro_excluding_raw0":float(np.mean(dice_ex_raw0)) if dice_ex_raw0 else None}

def loaders(records:list[SliceRecord],num_classes:int,label_map:dict[int,int]|None):
    split={s:[r for r in records if r.split==s] for s in ["train","val","test"]}
    for s,rs in split.items():
        if not rs: raise ValueError(f"Split vacio {s}")
    return tuple(DataLoader(Segmentation2DDataset(split[s],num_classes,label_map,s=="train"),batch_size=CFG.batch_size,shuffle=(s=="train"),num_workers=CFG.num_workers,pin_memory=torch.cuda.is_available()) for s in ["train","val","test"])

def train_one_model(*,plane:str,model_key:str,records:list[SliceRecord],classes:list[str],label_map:dict[int,int]|None,output_name:str,raw0_boost:float=1.0)->dict[str,Any]:
    num_classes=len(classes); train_loader,val_loader,test_loader=loaders(records,num_classes,label_map)
    weights=estimate_weights([r for r in records if r.split=="train"],num_classes,label_map,{1:raw0_boost} if plane=="axial" and raw0_boost>1 else None).to(DEVICE)
    model=build_model(plane,num_classes,CFG.base_channels).to(DEVICE); ce=nn.CrossEntropyLoss(weight=weights)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG.lr,weight_decay=1e-4); sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",factor=0.5,patience=4)
    best=-1.0; best_state=None; bad=0; history=[]
    for epoch in range(1,CFG.max_epochs+1):
        model.train(); losses=[]
        for x,y in train_loader:
            x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad(set_to_none=True); logits=model(x)
            loss=0.5*ce(logits,y)+0.5*dice_loss(logits,y,num_classes); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); losses.append(float(loss.detach().cpu()))
        vm=evaluate(model,val_loader,num_classes)
        monitor_metric="dice_macro_excluding_raw0" if plane=="axial" else "dice_macro_no_bg"
        score=vm.get(monitor_metric) or 0.0
        sch.step(score)
        history.append({"epoch":epoch,"train_loss":float(np.mean(losses)),"val":vm,"monitor_metric":monitor_metric,"monitor_score":score}); print(epoch, history[-1]["train_loss"], monitor_metric, score)
        if score>best: best=score; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=CFG.patience: print("early stopping"); break
    model.load_state_dict(best_state,strict=True); tm=evaluate(model,test_loader,num_classes)
    ckpt={"model_state_dict":model.state_dict(),"num_classes":num_classes,"base_channels":CFG.base_channels,"target_size":CFG.target_size,"class_weights":weights.detach().cpu().tolist(),"classes":classes,"metrics":{"best_val_monitor_score":best,"early_stopping_metric":("dice_macro_excluding_raw0" if plane=="axial" else "dice_macro_no_bg"),"test":tm,"history":history,"quality_gate_target_dice_macro_no_bg":0.70,"quality_gate_note":"Axial reports dice_macro_no_bg and dice_macro_excluding_raw0; gate final se define sobre split curado E9 por paciente."},"model_key":model_key,"plane":plane,"seed":CFG.seed,"created_at_unix":int(time.time())}
    if plane=="sagittal": ckpt["label_group_mapping"]={str(k):int(v) for k,v in (label_map or {}).items()}
    if plane=="axial": ckpt["label_map"]={str(k):int(v) for k,v in (label_map or {0:0,1:1,2:2,3:3,4:4,5:5}).items()}; ckpt["label_map_note"]="E9 curado esperado en 0..5; pairing crudo usa 250,0,50,100,150,200 -> 0..5."
    out=CFG.output_dir/output_name; torch.save(ckpt,out)
    loaded=torch.load(out,map_location="cpu",weights_only=False); strict_model,meta=build_checkpoint_model(model_key,loaded); strict_model.eval()
    with torch.inference_mode(): shape=list(strict_model(torch.randn(1,1,*CFG.target_size)).shape)
    digest=sha256(out.read_bytes()).hexdigest(); result={"artifact":str(out),"sha256":digest,"metrics":tm,"strict_load_smoke_shape":shape,"runtime_meta":meta}
    (CFG.output_dir/f"{out.stem}.metrics.json").write_text(json.dumps(result,indent=2),encoding="utf-8"); print(json.dumps(result,indent=2)); return result

## Entrenamiento sagital SPIDER

Carga mapping real, arma split por paciente, congela CSV y entrena. Descomentar la llamada en Colab/GPU.

In [ ]:
RUN_SAGITTAL = os.getenv("RUN_SAGITTAL", "1") == "1"
if RUN_SAGITTAL:
    spider_records, spider_label_map = build_spider_records()
    summarize_records(spider_records, "SPIDER sagittal")
    freeze_records(spider_records, CFG.output_dir / "spider_sagittal_patient_split.csv")
    sagittal_result = train_one_model(
        plane="sagittal",
        model_key="sagittal_spider",
        records=spider_records,
        classes=SAGITTAL_CLASSES,
        label_map=spider_label_map,
        output_name="sagittal_spider_multiclass_final_best.pt",
    )
else:
    sagittal_result = None


## Entrenamiento axial ALKAFRI/Sudirman

Preferir split curado E9 por paciente, con identidad de labels 0..5. Si se usa pairing crudo, activar `AXIAL_RAW_PAIRING_MODE=1`. `raw_0` se pondera con boost configurable por `AXIAL_RAW0_WEIGHT_BOOST`.

In [ ]:
RUN_AXIAL = os.getenv("RUN_AXIAL", "1") == "1"
if RUN_AXIAL:
    axial_records, axial_label_map = build_axial_records()
    summarize_records(axial_records, "ALKAFRI axial")
    freeze_records(axial_records, CFG.output_dir / "alkafri_axial_patient_split.csv")
    axial_result = train_one_model(
        plane="axial",
        model_key="axial_t2_alkafri",
        records=axial_records,
        classes=AXIAL_CLASSES,
        label_map=axial_label_map,
        output_name="axial_t2_alkafri_final_best.pt",
        raw0_boost=float(os.getenv("AXIAL_RAW0_WEIGHT_BOOST", "3.0")),
    )
else:
    axial_result = None


## Resumen de quality gate y materializacion

Despues de entrenar ambos modelos, ejecutar el resumen. No inventar metricas: copiar solo los numeros reales emitidos por el notebook.

In [ ]:
def quality_gate_summary(results:dict[str,dict[str,Any]])->pd.DataFrame:
    rows=[]
    for plane,result in results.items():
        m=result["metrics"]
        rows.append({"plane":plane,"artifact":Path(result["artifact"]).name,"sha256":result["sha256"],"dice_macro_no_bg":m.get("dice_macro_no_bg"),"iou_macro_no_bg":m.get("iou_macro_no_bg"),"dice_macro_excluding_raw0":m.get("dice_macro_excluding_raw0"),"passes_0_70_no_bg":(m.get("dice_macro_no_bg") or 0)>=0.70,"passes_0_70_excluding_raw0":(m.get("dice_macro_excluding_raw0") or 0)>=0.70})
    df=pd.DataFrame(rows); df.to_csv(CFG.output_dir/"final_training_quality_gate_summary.csv",index=False); return df

results = {k: v for k, v in {"sagittal": sagittal_result, "axial": axial_result}.items() if v is not None}
summary = quality_gate_summary(results)
display(summary)

## Instrucciones post-entrenamiento

1. Copiar los `.pt` generados desde `CFG.output_dir` a `models/final/` del repo local.
2. Verificar SHA-256 contra los `.metrics.json`.
3. Actualizar manifests y model cards con las metricas reales.
4. Ejecutar validaciones locales:

```powershell
$env:PYTHONPATH="ai_service"
.\.venv\Scripts\python.exe -m pytest ai_service	ests	est_model_manifest_validation.py ai_service	ests	est_sagittal_final_checkpoint_load.py ai_service	ests	est_axial_final_checkpoint_load.py -q
```

5. No commitear `.pt`, datasets, mascaras ni outputs pesados.